In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [10]:
filename = "001.queryVentes.csv"
df = pd.read_csv(filename)

In [11]:
df.shape

(85674, 20)

In [12]:
df.head(3)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),TEMPERATURE,METEO DU JOUR (MOYENNE),METEO DU MOIS (MOYENNE)
0,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,12:04:22.555,DONE,3.583788e+12,TONGS FEMME 100 NOIR,1,NaN,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,23.1,NaN,NaN
1,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,18:32:38.901,DONE,3.608439e+12,CASQUETTE ENFANT -MH100,1,NaN,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,23.1,NaN,NaN
2,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-11,12:11:15.59,DONE,3.583788e+12,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,NaN,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,24.6,NaN,NaN


In [13]:
df.dtypes

NOM BOUTIQUE                    object
OPERATEUR                       object
MACHINE                         object
DATE                            object
HEURE                           object
STATUT                          object
CODE EAN                       float64
NOM DU PRODUIT                  object
QUANTITE                         int64
PRIX HT                        float64
VAT                            float64
PRIX TTC                       float64
TYPE                            object
GAMME                           object
MARQUE                          object
FOURNISSEUR                     object
ORDER ID (TICKET DE CAISSE)      int64
TEMPERATURE                    float64
METEO DU JOUR (MOYENNE)        float64
METEO DU MOIS (MOYENNE)        float64
dtype: object

In [ ]:
dt_col = "DATE"
tm_col = "HEURE"
dttm_col = "DATETIME"

df[dttm_col] = pd.to_datetime(df[dt_col] + " " + df[tm_col], format = "mixed")

df["ANNEE"] = df[dttm_col].dt.year
df["MOIS_D_ANNEE"] = df[dttm_col].dt.month
df["JOUR_DU_MOIS"] = df[dttm_col].dt.day
df["JOUR_DE_SEMAINE"] = df[dttm_col].dt.dayofweek
df["JOUR_D_ANNEE"] = df[dttm_col].dt.dayofyear
df["SEMAINE_D_ANNEE"] = df[dttm_col].dt.isocalendar().week.astype(int)
df["IS_WEEKEND"] = df["JOUR_DE_SEMAINE"].isin([5, 6]).astype(int)
df["HEURE_DU_JOUR"] = df[dttm_col].dt.hour
df.head(3)

df["JOUR_DU_MOIS"].nunique()

for i in range(2, 30):
    df[f"IS_JOUR_DU_MOIS_GEQ_{i}"] = (df["JOUR_DU_MOIS"] >= i).astype(int)

df.head(5)

,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,...,IS_JOUR_DU_MOIS_GEQ_20,IS_JOUR_DU_MOIS_GEQ_21,IS_JOUR_DU_MOIS_GEQ_22,IS_JOUR_DU_MOIS_GEQ_23,IS_JOUR_DU_MOIS_GEQ_24,IS_JOUR_DU_MOIS_GEQ_25,IS_JOUR_DU_MOIS_GEQ_26,IS_JOUR_DU_MOIS_GEQ_27,IS_JOUR_DU_MOIS_GEQ_28,IS_JOUR_DU_MOIS_GEQ_29
0,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,12:04:22.555,DONE,3.583788e+12,TONGS FEMME 100 NOIR,1,NaN,...,0,0,0,0,0,0,0,0,0,0
1,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,18:32:38.901,DONE,3.608439e+12,CASQUETTE ENFANT -MH100,1,NaN,...,0,0,0,0,0,0,0,0,0,0
2,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-11,12:11:15.59,DONE,3.583788e+12,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,NaN,...,0,0,0,0,0,0,0,0,0,0
3,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-12,12:50:46.708,DONE,3.583788e+12,LUNETTES DE NATATION VERRES CLAIRS XBASE TAILL...,1,NaN,...,0,0,0,0,0,0,0,0,0,0
4,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-12,17:00:42.96,DONE,3.583787e+12,BOARDSHORT HENDAIA ECO NT BLEU,1,NaN,...,0,0,0,0,0,0,0,0,0,0


In [15]:
class Prep():
    def __init__(self, filepath : str):
        self.filepath = filepath

        self._df = None

    @property
    def df(self):
        if(self._df is None):
            self._df = pd.read_csv(self.filepath)
        return self._df
    

    
    


p = Prep("001.queryVentes.csv")
p.df["DATE"]

0        2023-08-10
1        2023-08-10
2        2023-08-11
3        2023-08-12
4        2023-08-12
            ...    
85669    2026-04-07
85670    2026-04-07
85671    2026-04-07
85672    2026-04-07
85673    2026-04-07
Name: DATE, Length: 85674, dtype: object

In [16]:
ORDER_COL = "ORDER ID (TICKET DE CAISSE)"
PRODUCT_COL = "GAMME"
df = (
    df[df["STATUT"].str.upper() == "DONE"][[ORDER_COL, PRODUCT_COL]]
    .dropna()
    .drop_duplicates(subset = [ORDER_COL, PRODUCT_COL])
)


In [17]:
basket = (
    df.assign(value=1)
    .pivot_table(
        index=ORDER_COL,
        columns=PRODUCT_COL,
        values="value",
        aggfunc="max",
        fill_value=0
    )          
    .astype(bool)
)

In [18]:
basket

GAMME,#REF!,ACCESSOIRES,ALCOOL,COSMETIQUE,FOOD SALEE,FOOD SUCREE,JEUX / ENFANTS,PAP,SANS ALCOOL,SOS,SOUVENIRS
ORDER ID (TICKET DE CAISSE),,,,,,,,,,,
1,False,False,False,False,False,True,False,False,True,False,False
2,False,False,True,False,False,False,False,False,True,False,False
3,False,False,True,False,False,False,False,False,True,False,False
4,False,False,False,False,False,True,False,False,False,False,False
6,False,False,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
56586,False,False,False,False,False,False,False,False,True,False,False
56587,False,False,False,False,False,False,False,False,True,False,False
56588,False,False,False,False,True,False,False,False,False,False,False


In [19]:
frequent_itemsets = apriori(
    basket,
    min_support=0.01,   # à ajuster selon ton volume
    use_colnames=True
)

In [20]:
frequent_itemsets.shape

(8, 2)

In [21]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

In [23]:
rules.shape

(2, 14)

In [24]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(FOOD SALEE),(FOOD SUCREE),0.189771,0.187677,0.045478,0.239647,1.276911,1.0,0.009862,1.068350,0.267653,0.136995,0.063977,0.240984
1,(FOOD SUCREE),(FOOD SALEE),0.187677,0.189771,0.045478,0.242321,1.276911,1.0,0.009862,1.069356,0.266963,0.136995,0.064858,0.240984


In [25]:
rules[
    (rules["confidence"] >= 0.20) &
    (rules["lift"] >= 1.20)
].sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(FOOD SUCREE),(FOOD SALEE),0.187677,0.189771,0.045478,0.242321,1.276911,1.0,0.009862,1.069356,0.266963,0.136995,0.064858,0.240984
0,(FOOD SALEE),(FOOD SUCREE),0.189771,0.187677,0.045478,0.239647,1.276911,1.0,0.009862,1.068350,0.267653,0.136995,0.063977,0.240984
